In [1]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

dataset = "xdxd003/ff-c23"

# List all files in the dataset (this may take a moment)
files = api.dataset_list_files(dataset).files

print(f"Total files in dataset: {len(files)}")

Total files in dataset: 20


In [3]:
# Look at the first file object's actual attributes
print(dir(files[0]))

['__class__', '__contains__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_columns', '_creation_date', '_dataset_ref', '_description', '_fields', '_file_type', '_freeze', '_get_field', '_is_frozen', '_name', '_owner_ref', '_ref', '_total_bytes', '_url', 'body_fields', 'columns', 'creation_date', 'dataset_ref', 'description', 'endpoint', 'endpoint_path', 'file_type', 'from_dict', 'from_json', 'method', 'name', 'owner_ref', 'prepare_from', 'ref', 'to_dict', 'to_field_map', 'to_json', 'total_bytes', 'url']


In [4]:
for f in files:
    print(f.name, f.total_bytes)

FaceForensics++_C23/DeepFakeDetection/01_02__meeting_serious__YVGY8LOK.mp4 6745903
FaceForensics++_C23/DeepFakeDetection/01_02__outside_talking_still_laughing__YVGY8LOK.mp4 5290755
FaceForensics++_C23/DeepFakeDetection/01_02__talking_against_wall__YVGY8LOK.mp4 3471524
FaceForensics++_C23/DeepFakeDetection/01_02__walk_down_hall_angry__YVGY8LOK.mp4 1164071
FaceForensics++_C23/DeepFakeDetection/01_02__walking_down_indoor_hall_disgust__YVGY8LOK.mp4 12640206
FaceForensics++_C23/DeepFakeDetection/01_03__hugging_happy__ISF9SP4G.mp4 9123749
FaceForensics++_C23/DeepFakeDetection/01_03__kitchen_pan__JZUXXFRB.mp4 3485507
FaceForensics++_C23/DeepFakeDetection/01_03__podium_speech_happy__480LQD1C.mp4 5056283
FaceForensics++_C23/DeepFakeDetection/01_03__talking_against_wall__JZUXXFRB.mp4 3489591
FaceForensics++_C23/DeepFakeDetection/01_04__hugging_happy__GBC7ZGDP.mp4 8246897
FaceForensics++_C23/DeepFakeDetection/01_04__meeting_serious__0XUW13RW.mp4 6563740
FaceForensics++_C23/DeepFakeDetection/01_04

In [5]:
help(api.dataset_list_files)

Help on method dataset_list_files in module kaggle.api.kaggle_api_extended:

dataset_list_files(dataset, page_token=None, page_size=20) method of kaggle.api.kaggle_api_extended.KaggleApi instance
    Lists files for a dataset.
    
    Args:
        dataset: The string identifier of the dataset, in the format [owner]/[dataset-name].
        page_token: The page token for pagination.
        page_size: The number of items per page.



In [6]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

dataset = "xdxd003/ff-c23"

all_files = []
page_token = None

while True:
    result = api.dataset_list_files(dataset, page_token=page_token, page_size=500)
    all_files.extend(result.files)
    page_token = result.next_page_token
    if not page_token:
        break

print(f"Total files collected: {len(all_files)}")

Total files collected: 7010


In [7]:
original_files = [f.name for f in all_files if f.name.startswith("FaceForensics++_C23/original/")]
deepfakes_files = [f.name for f in all_files if f.name.startswith("FaceForensics++_C23/Deepfakes/")]

print(f"Original (real) videos found: {len(original_files)}")
print(f"Deepfakes videos found: {len(deepfakes_files)}")

# peek at a few filenames to confirm naming pattern
print(original_files[:5])
print(deepfakes_files[:5])

Original (real) videos found: 1000
Deepfakes videos found: 1000
['FaceForensics++_C23/original/000.mp4', 'FaceForensics++_C23/original/001.mp4', 'FaceForensics++_C23/original/002.mp4', 'FaceForensics++_C23/original/003.mp4', 'FaceForensics++_C23/original/004.mp4']
['FaceForensics++_C23/Deepfakes/000_003.mp4', 'FaceForensics++_C23/Deepfakes/001_870.mp4', 'FaceForensics++_C23/Deepfakes/002_006.mp4', 'FaceForensics++_C23/Deepfakes/003_000.mp4', 'FaceForensics++_C23/Deepfakes/004_982.mp4']


In [9]:
import random
import os
from tqdm import tqdm

# Reproducibility - same random sample every time we run this
random.seed(42)

# How many videos we want from each class
SAMPLE_SIZE = 1000

sampled_original = random.sample(original_files, SAMPLE_SIZE)
sampled_deepfakes = random.sample(deepfakes_files, SAMPLE_SIZE)

print(f"Selected {len(sampled_original)} real videos")
print(f"Selected {len(sampled_deepfakes)} fake videos")

Selected 1000 real videos
Selected 1000 fake videos


Real download loop is in the cell given below

In [10]:
# Destination folders (temporary staging area before we split into train/val/test in Phase 8)
real_dest = "D:/Deepfake-detection/datasets/raw_original"
fake_dest = "D:/Deepfake-detection/datasets/raw_deepfakes"

os.makedirs(real_dest, exist_ok=True)
os.makedirs(fake_dest, exist_ok=True)

print("Downloading real (original) videos...")
for filename in tqdm(sampled_original):
    api.dataset_download_file(dataset, file_name=filename, path=real_dest)

print("Downloading fake (Deepfakes) videos...")
for filename in tqdm(sampled_deepfakes):
    api.dataset_download_file(dataset, file_name=filename, path=fake_dest)

print("Done!")

  0%|          | 0/1000 [00:00<?, ?it/s]

Dataset URL: https://www.kaggle.com/datasets/xdxd003/ff-c23


  0%|          | 1/1000 [01:33<25:51:29, 93.18s/it]

Dataset URL: https://www.kaggle.com/datasets/xdxd003/ff-c23


  0%|          | 2/1000 [03:02<25:15:22, 91.10s/it]

...

Dataset URL: https://www.kaggle.com/datasets/xdxd003/ff-c23


 66%|██████▌   | 662/1000 [57:41<1:20:08, 14.23s/it]

Dataset URL: https://www.kaggle.com/datasets/xdxd003/ff-c23


 66%|██████▌   | 662/1000 [58:04<29:39,  5.26s/it]  


SSLError: HTTPSConnectionPool(host='api.kaggle.com', port=443): Max retries exceeded with url: /v1/datasets.DatasetApiService/DownloadDataset (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1016)')))

In [11]:
import os

existing_real = set(os.listdir(real_dest))
print(f"Already downloaded: {len(existing_real)} real videos")

Already downloaded: 662 real videos


In [12]:
import time

def download_with_retry(filenames, dest_folder, max_retries=3):
    existing = set(os.listdir(dest_folder))
    
    for filename in tqdm(filenames):
        local_name = filename.split("/")[-1]
        
        if local_name in existing:
            continue
        
        attempt = 0
        while attempt < max_retries:
            try:
                api.dataset_download_file(dataset, file_name=filename, path=dest_folder)
                break
            except Exception as e:
                attempt += 1
                if attempt == max_retries:
                    print(f"FAILED after {max_retries} attempts: {filename} -> {e}")
                else:
                    time.sleep(3)

In [13]:
print("Downloading real (original) videos...")
download_with_retry(sampled_original, real_dest)

print("Downloading fake (Deepfakes) videos...")
download_with_retry(sampled_deepfakes, fake_dest)

print("Done!")

  0%|          | 0/1000 [00:00<?, ?it/s]

Dataset URL: https://www.kaggle.com/datasets/xdxd003/ff-c23


 66%|██████▋   | 663/1000 [00:22<00:11, 29.46it/s]

Dataset URL: https://www.kaggle.com/datasets/xdxd003/ff-c23
Dataset URL: https://www.kaggle.com/datasets/xdxd003/ff-c23


 66%|██████▋   | 665/1000 [00:38<00:22, 14.63it/s]

...

Dataset URL: https://www.kaggle.com/datasets/xdxd003/ff-c23


100%|█████████▉| 999/1000 [4:08:55<00:04,  4.41s/it]

Dataset URL: https://www.kaggle.com/datasets/xdxd003/ff-c23


100%|██████████| 1000/1000 [4:08:59<00:00, 14.94s/it]

Done!


In [14]:
import os

real_count = len(os.listdir(real_dest))
fake_count = len(os.listdir(fake_dest))

print(f"Real videos on disk: {real_count}")
print(f"Fake videos on disk: {fake_count}")

Real videos on disk: 1000
Fake videos on disk: 999


In [15]:
downloaded_fake_names = set(os.listdir(fake_dest))

missing = []
for f in deepfakes_files:
    local_name = f.split("/")[-1]
    if local_name not in downloaded_fake_names:
        missing.append(f)

print(f"Missing: {missing}")

Missing: ['FaceForensics++_C23/Deepfakes/591_605.mp4']


In [16]:
if missing:
    api.dataset_download_file(dataset, file_name=missing[0], path=fake_dest)
    print("Downloaded the missing file.")

Dataset URL: https://www.kaggle.com/datasets/xdxd003/ff-c23
Downloaded the missing file.


In [17]:
print(f"Fake videos on disk now: {len(os.listdir(fake_dest))}")

Fake videos on disk now: 1000
